# DNABERT-2 + RCCR: Does RC Consistency Regularization Reduce the Geometric Tax?

**Motivation**: Ma (2025) introduces Reverse-Complement Consistency Regularization (RCCR),
a lightweight fine-tuning objective that penalizes divergence between a model's output on
sequence `x` and its reverse complement `RC(x)`. Applied to exactly the backbones we
evaluate: Nucleotide Transformer, HyenaDNA, and DNABERT-2.

**Experiment**: We take DNABERT-2 (117M), apply RCCR fine-tuning (self-supervised, no
labels needed), then run our full Shesha harness + RC texture test on both baseline
and RCCR-finetuned versions.

**Key Question**: Does post-hoc RC consistency regularization reduce the geometric tax
on a discrete CE backbone?

- If RCCR reduces the RC gap but doesn't fix Procrustes stability → tax is deeper than symmetry violation
- If RCCR fixes both → we've found a mitigation, paper is stronger

**Reference**: Ma, M. (2025). *Reverse-Complement Consistency for DNA Language Models*. arXiv:2509.18529

---

In [ ]:
# Install Dependencies

print("Installing core dependencies...")
!pip install -q torch transformers shesha-geometry matplotlib seaborn pandas einops triton

import transformers, torch
print(f"transformers {transformers.__version__}")
print(f"torch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# --- Patch transformers 5.x for DNABERT-2 compatibility ---
from transformers import PretrainedConfig
import transformers.modeling_utils as _mu

_config_defaults = {
    'pad_token_id': None, 'bos_token_id': None, 'eos_token_id': None,
    'is_decoder': False, 'add_cross_attention': False,
    'chunk_size_feed_forward': 0, 'is_encoder_decoder': False,
    'tie_word_embeddings': True,
}
for attr, default in _config_defaults.items():
    if not hasattr(PretrainedConfig, attr):
        setattr(PretrainedConfig, attr, default)
        print(f"Patched config default: {attr}={default}")

if not getattr(_mu, '_dnabert2_patched', False):
    orig_mark = _mu.PreTrainedModel.mark_tied_weights_as_initialized
    def safe_mark(self):
        if not hasattr(self, 'all_tied_weights_keys'):
            self.all_tied_weights_keys = {}
        return orig_mark(self)
    _mu.PreTrainedModel.mark_tied_weights_as_initialized = safe_mark

    orig_finalize = _mu.PreTrainedModel._finalize_load_state_dict
    @staticmethod
    def safe_finalize(model, load_config, load_info):
        if not hasattr(model, 'all_tied_weights_keys'):
            model.all_tied_weights_keys = {}
        orig_tie = model.tie_weights
        def _patched_tie(missing_keys=None, recompute_mapping=False, **kwargs):
            return orig_tie()
        model.tie_weights = _patched_tie
        return orig_finalize(model, load_config, load_info)
    _mu.PreTrainedModel._finalize_load_state_dict = safe_finalize
    _mu._dnabert2_patched = True
    print("Patched: all_tied_weights_keys / tie_weights")

from contextlib import contextmanager

@contextmanager
def _no_meta_init(**kwargs):
    yield

try:
    import accelerate
    accelerate.init_empty_weights = _no_meta_init
    print("Patched: accelerate.init_empty_weights")
except ImportError:
    pass

for mod_name in ['transformers.modeling_utils', 'transformers.integrations.accelerate']:
    import sys
    mod = sys.modules.get(mod_name)
    if mod and hasattr(mod, 'init_empty_weights'):
        mod.init_empty_weights = _no_meta_init
        print(f"Patched: {mod_name}.init_empty_weights")

print("\nDone!")

In [ ]:
# Configuration

import sys, os
sys.path.insert(0, '.')
import numpy as np

PHASE = 'full'  # 'quick' for a fast sanity check, 'full' for complete results

OUTPUT_BASE = './results/dnabert2_rccr_experiment/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR = OUTPUT_BASE + 'cache'

MODEL_NAME = 'zhihan1996/DNABERT-2-117M'

CONFIG = {
    'quick': {
        'n_sequences': 500,
        'seq_length': 512,
        'batch_size': 8,
        'n_bootstrap': 0,
        'snp_rates': [0.01, 0.05, 0.10],
        # RCCR fine-tuning
        'rccr_epochs': 5,
        'rccr_lr': 2e-5,
        'rccr_lambda': 1.0,
        'rccr_batch_size': 4,
        'rccr_n_seqs': 500,
    },
    'full': {
        'n_sequences': 5000,
        'seq_length': 512,
        'batch_size': 8,
        'n_bootstrap': 5,
        'snp_rates': [0.01, 0.02, 0.05, 0.10],
        # RCCR fine-tuning
        'rccr_epochs': 10,
        'rccr_lr': 2e-5,
        'rccr_lambda': 1.0,
        'rccr_batch_size': 4,
        'rccr_n_seqs': 2000,
    },
}

config = CONFIG[PHASE]
print(f"Phase: {PHASE.upper()}")
print(f"Eval sequences: {config['n_sequences']}")
print(f"RCCR epochs: {config['rccr_epochs']}, lambda: {config['rccr_lambda']}")

In [ ]:
# DNA Utilities

DNA_BASES = ['A', 'C', 'G', 'T']
COMPLEMENT = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}

def generate_dna_sequences(n_sequences, seq_length, seed=320):
    rng = np.random.default_rng(seed)
    sequences = [
        ''.join(rng.choice(DNA_BASES, size=seq_length))
        for _ in range(n_sequences)
    ]
    print(f"Generated {len(sequences)} DNA sequences (length={seq_length})")
    return sequences

def reverse_complement(sequence):
    return ''.join(COMPLEMENT.get(b, b) for b in reversed(sequence))

def mutate_dna(sequence, mutation_rate, rng):
    seq = list(sequence)
    n_mutations = max(1, int(len(seq) * mutation_rate))
    positions = rng.choice(len(seq), size=n_mutations, replace=False)
    for pos in positions:
        original = seq[pos]
        alternatives = [b for b in DNA_BASES if b != original]
        seq[pos] = rng.choice(alternatives)
    return ''.join(seq)

from dataclasses import dataclass, field

@dataclass
class PerturbedSet:
    name: str
    category: str
    sequences: list
    params: dict = field(default_factory=dict)
    description: str = ''

class DNAPerturbationSuite:
    def __init__(self, seed=320, snp_rates=None, include_rc=True):
        self.rng = np.random.default_rng(seed)
        self.snp_rates = snp_rates or [0.01, 0.02, 0.05, 0.10]
        self.include_rc = include_rc

    def run_all(self, sequences):
        results = {}
        for rate in self.snp_rates:
            name = f"snp_{int(rate * 100)}pct"
            perturbed = [mutate_dna(s, rate, self.rng) for s in sequences]
            results[name] = PerturbedSet(
                name=name, category='snp', sequences=perturbed,
                params={'mutation_rate': rate},
                description=f'SNP mutations at {rate*100:.0f}% rate',
            )
        if self.include_rc:
            results['reverse_complement'] = PerturbedSet(
                name='reverse_complement', category='rc',
                sequences=[reverse_complement(s) for s in sequences],
                params={}, description='Reverse complement transformation',
            )
        return results

suite = DNAPerturbationSuite(seed=320, snp_rates=config['snp_rates'])
print("Perturbation suite ready")

In [ ]:
# Load DNABERT-2 Model

import torch
import gc
from transformers import AutoTokenizer, AutoConfig
from huggingface_hub import hf_hub_download

def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def _patch_flash_attn_triton_globally():
    """Find and patch ALL copies of flash_attn_triton.py in the HF cache.
    
    Triton >= 3.x removed trans_a/trans_b kwargs from tl.dot().
    DNABERT-2's bundled flash_attn_triton.py uses both:
      - trans_b=True in the forward pass
      - trans_a=True in the backward pass
    We replace them with explicit tl.trans() calls.
    """
    import glob, os, re
    
    # Search common HF cache locations for ALL copies
    search_roots = [
        os.path.expanduser("~/.cache/huggingface"),
        "/root/.cache/huggingface",
        "/tmp",
    ]
    
    patched_files = []
    for root in search_roots:
        if not os.path.exists(root):
            continue
        for fpath in glob.glob(os.path.join(root, "**", "flash_attn_triton.py"), recursive=True):
            with open(fpath, 'r') as f:
                source = f.read()
            
            changed = False
            if 'trans_b' in source:
                source = re.sub(
                    r'tl\.dot\(([^,]+),\s*([^,]+),\s*trans_b\s*=\s*True\)',
                    r'tl.dot(\1, tl.trans(\2))', source
                )
                changed = True
            if 'trans_a' in source:
                source = re.sub(
                    r'tl\.dot\(([^,]+),\s*([^,]+),\s*trans_a\s*=\s*True\)',
                    r'tl.dot(tl.trans(\1), \2)', source
                )
                changed = True
            
            if changed:
                with open(fpath, 'w') as f:
                    f.write(source)
                patched_files.append(fpath)
    
    # Also clear any cached Triton compilations
    triton_cache = os.path.expanduser("~/.triton/cache")
    if os.path.exists(triton_cache):
        import shutil
        shutil.rmtree(triton_cache, ignore_errors=True)
        print("  Cleared Triton compilation cache")
    
    return patched_files

def load_dnabert2(model_name):
    """Load DNABERT-2 model and tokenizer. Returns (model, tokenizer, n_params_m)."""
    import sys, os, re
    from transformers.dynamic_module_utils import get_class_from_dynamic_module

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Loading {model_name} on {device}...")

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    cfg = AutoConfig.from_pretrained(model_name, trust_remote_code=True)

    # Patch ALL copies of flash_attn_triton.py for Triton >= 3.x
    patched = _patch_flash_attn_triton_globally()
    if patched:
        print(f"  Patched {len(patched)} flash_attn_triton.py file(s):")
        for p in patched:
            print(f"    {p}")
        # Clear any already-imported modules so they reload from patched files
        for key in list(sys.modules.keys()):
            if 'flash_attn_triton' in key or 'bert_layers' in key:
                del sys.modules[key]
    else:
        print("  No flash_attn_triton.py files needed patching")

    auto_map = getattr(cfg, 'auto_map', {})
    class_ref = auto_map.get('AutoModel', auto_map.get('AutoModelForMaskedLM', 'bert_layers.BertModel'))
    BertModel = get_class_from_dynamic_module(class_ref, model_name)

    model = BertModel(cfg)
    weight_file = hf_hub_download(model_name, 'pytorch_model.bin')
    state_dict = torch.load(weight_file, map_location='cpu', weights_only=False)
    model.load_state_dict(state_dict, strict=False)
    model = model.to(device).eval()

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"  Params: {n_params:.1f}M")

    max_length = tokenizer.model_max_length
    if max_length > 100000:
        max_length = 512

    return model, tokenizer, n_params, max_length, device

model, tokenizer, n_params_m, max_length, device = load_dnabert2(MODEL_NAME)
print(f"Model loaded: {n_params_m:.1f}M params, max_length={max_length}")


In [ ]:
# Embedding Function

def make_embed_fn(model, tokenizer, max_length, device, batch_size=8):
    """Create embedding function from model."""
    @torch.no_grad()
    def embed(sequences):
        embeddings = []
        n_batches = (len(sequences) + batch_size - 1) // batch_size
        for i in range(0, len(sequences), batch_size):
            batch = sequences[i:i + batch_size]
            batch_idx = i // batch_size + 1
            if batch_idx % 20 == 0 or batch_idx == n_batches:
                print(f"    Batch {batch_idx}/{n_batches}", end='\r')
            tokens = tokenizer(
                batch, return_tensors='pt', padding=True,
                truncation=True, max_length=max_length,
            )
            tokens = {k: v.to(device) for k, v in tokens.items()}
            outputs = model(**tokens)
            hidden = outputs[0]
            attention_mask = tokens['attention_mask'].unsqueeze(-1)
            pooled = (hidden * attention_mask).sum(dim=1) / attention_mask.sum(dim=1).clamp(min=1)
            embeddings.append(pooled.cpu().numpy())
        print()
        return np.concatenate(embeddings, axis=0)
    return embed

print("Embedding function ready")

In [ ]:
# RCCR Fine-Tuning
#
# Implements Ma (2025) RCCR: self-supervised consistency regularization.
# Loss = MSE( embed(x), embed(RC(x)) )  -- no labels needed.
#
# For embedding-level RCCR (unsupervised), we minimize the distance between
# the pooled representation of a sequence and its reverse complement.
# This is the "representation consistency" variant -- directly applicable
# to our geometric stability evaluation.

import copy
import time

def rccr_finetune(model, tokenizer, max_length, device, config):
    """Apply RCCR fine-tuning to enforce RC consistency in embedding space.
    
    L_RCCR = lambda * MSE(pool(f(x)), pool(f(RC(x))))
    
    This is the unsupervised/self-supervised variant: we don't need task labels,
    just DNA sequences. We minimize the L2 distance between embeddings of
    a sequence and its reverse complement.
    """
    print("="*60)
    print("RCCR FINE-TUNING")
    print("="*60)
    
    # Generate training sequences (separate from eval sequences)
    rng = np.random.default_rng(seed=999)
    n_train = config['rccr_n_seqs']
    train_seqs = [
        ''.join(rng.choice(DNA_BASES, size=config['seq_length']))
        for _ in range(n_train)
    ]
    print(f"Training sequences: {n_train}")
    print(f"Epochs: {config['rccr_epochs']}, LR: {config['rccr_lr']}, Lambda: {config['rccr_lambda']}")
    
    # Deep copy the model for RCCR fine-tuning
    model_rccr = copy.deepcopy(model)
    model_rccr.train()
    
    optimizer = torch.optim.AdamW(model_rccr.parameters(), lr=config['rccr_lr'], weight_decay=0.01)
    batch_size = config['rccr_batch_size']
    lam = config['rccr_lambda']
    
    loss_history = []
    t0 = time.time()
    
    for epoch in range(config['rccr_epochs']):
        epoch_losses = []
        # Shuffle training data
        indices = np.random.permutation(n_train)
        
        for i in range(0, n_train, batch_size):
            batch_idx = indices[i:i + batch_size]
            fwd_seqs = [train_seqs[j] for j in batch_idx]
            rc_seqs = [reverse_complement(s) for s in fwd_seqs]
            
            # Tokenize forward
            tok_fwd = tokenizer(
                fwd_seqs, return_tensors='pt', padding=True,
                truncation=True, max_length=max_length,
            )
            tok_fwd = {k: v.to(device) for k, v in tok_fwd.items()}
            
            # Tokenize RC
            tok_rc = tokenizer(
                rc_seqs, return_tensors='pt', padding=True,
                truncation=True, max_length=max_length,
            )
            tok_rc = {k: v.to(device) for k, v in tok_rc.items()}
            
            # Forward pass -- both sequences
            out_fwd = model_rccr(**tok_fwd)[0]  # (B, L, D)
            out_rc = model_rccr(**tok_rc)[0]
            
            # Mean-pool with attention mask
            mask_fwd = tok_fwd['attention_mask'].unsqueeze(-1)
            mask_rc = tok_rc['attention_mask'].unsqueeze(-1)
            pool_fwd = (out_fwd * mask_fwd).sum(1) / mask_fwd.sum(1).clamp(min=1)
            pool_rc = (out_rc * mask_rc).sum(1) / mask_rc.sum(1).clamp(min=1)
            
            # RCCR loss: MSE between forward and RC embeddings
            loss = lam * torch.nn.functional.mse_loss(pool_fwd, pool_rc)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_rccr.parameters(), 1.0)
            optimizer.step()
            
            epoch_losses.append(loss.item())
        
        mean_loss = np.mean(epoch_losses)
        loss_history.append(mean_loss)
        elapsed = time.time() - t0
        print(f"  Epoch {epoch+1}/{config['rccr_epochs']}: loss={mean_loss:.6f} ({elapsed:.0f}s)")
    
    model_rccr.eval()
    print(f"\nRCCR fine-tuning complete in {time.time()-t0:.0f}s")
    print(f"Loss: {loss_history[0]:.6f} -> {loss_history[-1]:.6f} ({(1 - loss_history[-1]/loss_history[0])*100:.1f}% reduction)")
    
    return model_rccr, loss_history

model_rccr, rccr_loss_history = rccr_finetune(model, tokenizer, max_length, device, config)

In [ ]:
# Evaluation Harness Setup

from evaluation_harness import StabilityHarness, ModelReport

harness = StabilityHarness(
    window_size=0,
    metric='cosine',
    n_splits=30,
    seed=320,
    max_samples=2500,
    n_bootstrap=config['n_bootstrap'],
)

print(f"Harness configured (bootstrap={config['n_bootstrap']})")

In [ ]:
# Run Evaluation on BOTH Models

import os
import pandas as pd

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# Generate evaluation sequences
eval_sequences = generate_dna_sequences(config['n_sequences'], config['seq_length'])

# Generate perturbations
print('Generating perturbations...')
perturbed_sets = suite.run_all(eval_sequences)

all_rows = []

for variant_name, variant_model in [('DNABERT2_baseline', model), ('DNABERT2_RCCR', model_rccr)]:
    print(f"\n{'='*60}")
    print(f"Evaluating: {variant_name}")
    print('='*60)
    
    embed_fn = make_embed_fn(variant_model, tokenizer, max_length, device, config['batch_size'])
    
    # Compute clean embeddings
    print('Computing clean embeddings...')
    embeddings_clean = embed_fn(eval_sequences)
    print(f'  Shape: {embeddings_clean.shape}')
    
    # Compute perturbed embeddings
    perturbed_embeddings = {}
    for pert_name, pset in perturbed_sets.items():
        print(f'  Embedding: {pert_name}...')
        perturbed_embeddings[pert_name] = embed_fn(pset.sequences)
    
    # Run Shesha evaluation
    print('Running Shesha evaluation...')
    report = harness.evaluate_all_perturbations(
        model_name=variant_name,
        embeddings_clean=embeddings_clean,
        perturbed_dict=perturbed_embeddings,
    )
    
    # Collect results
    for pert_name, metrics in report.perturbation_breakdown().items():
        all_rows.append({
            'model': variant_name,
            'perturbation': pert_name,
            'rdm_similarity': metrics.get('rdm_similarity_score', 0),
            'rdm_drift': metrics.get('rdm_drift_score', 0),
            'pert_stability': metrics.get('perturbation_stability_score', 0),
            'pert_magnitude': metrics.get('perturbation_magnitude', 0),
            'feature_split': metrics.get('feature_split_score', 0),
            'sample_split': metrics.get('sample_split_score', 0),
            'anchor_stability': metrics.get('anchor_stability_score', 0),
            'composite': metrics.get('composite_stability', 0),
        })
    
    summary = report.summary()
    print(f'\n  Composite: {summary["mean_composite_stability"]:.4f}')
    print(f'  RDM Sim:   {summary["mean_rdm_similarity_score"]:.4f}')
    print(f'  Pert Stab: {summary["mean_perturbation_stability_score"]:.4f}')

# Save results
df = pd.DataFrame(all_rows)
csv_path = f'{RESULTS_DIR}/dnabert2_rccr_{PHASE}_detailed.csv'
df.to_csv(csv_path, index=False)
print(f'\nSaved to {csv_path}')
print(df.to_string(index=False))

In [ ]:
# RC-Specific Texture Test
#
# Direct comparison: how much does the embedding change under RC transformation?
# This is the core "RC gap" measurement.

from scipy.spatial.distance import cosine as cosine_dist
from scipy.linalg import orthogonal_procrustes

def rc_texture_test(model_obj, tokenizer, max_length, device, sequences, name=''):
    """Measure RC consistency at the embedding level.
    
    Returns:
        rc_cosine_gaps: per-sequence cosine distance between fwd and RC embeddings
        procrustes_disparity: Procrustes distance between fwd and RC embedding matrices
    """
    embed_fn = make_embed_fn(model_obj, tokenizer, max_length, device, batch_size=8)
    
    rc_seqs = [reverse_complement(s) for s in sequences]
    
    print(f'  [{name}] Computing forward embeddings...')
    emb_fwd = embed_fn(sequences)
    print(f'  [{name}] Computing RC embeddings...')
    emb_rc = embed_fn(rc_seqs)
    
    # Per-sequence cosine gap
    cosine_gaps = []
    for i in range(len(sequences)):
        gap = cosine_dist(emb_fwd[i], emb_rc[i])
        cosine_gaps.append(gap)
    cosine_gaps = np.array(cosine_gaps)
    
    # Procrustes analysis: how well do the two embedding matrices align?
    # Center the matrices
    X = emb_fwd - emb_fwd.mean(axis=0)
    Y = emb_rc - emb_rc.mean(axis=0)
    
    # Normalize
    X_norm = X / np.linalg.norm(X, 'fro')
    Y_norm = Y / np.linalg.norm(Y, 'fro')
    
    # Optimal rotation
    R, scale = orthogonal_procrustes(X_norm, Y_norm)
    X_rotated = X_norm @ R
    
    # Procrustes disparity
    procrustes_disp = np.linalg.norm(X_rotated - Y_norm, 'fro') ** 2
    
    print(f'  [{name}] RC cosine gap: mean={cosine_gaps.mean():.6f}, std={cosine_gaps.std():.6f}')
    print(f'  [{name}] Procrustes disparity: {procrustes_disp:.6f}')
    
    return cosine_gaps, procrustes_disp

# Use a subset for the texture test
texture_seqs = eval_sequences[:min(500, len(eval_sequences))]

print('\n' + '='*60)
print('RC TEXTURE TEST')
print('='*60)

gaps_baseline, proc_baseline = rc_texture_test(
    model, tokenizer, max_length, device, texture_seqs, 'Baseline'
)
gaps_rccr, proc_rccr = rc_texture_test(
    model_rccr, tokenizer, max_length, device, texture_seqs, 'RCCR'
)

print(f'\n{"="*60}')
print(f'RC TEXTURE TEST SUMMARY')
print(f'{"="*60}')
print(f'{"Metric":<30} {"Baseline":>12} {"RCCR":>12} {"Change":>12}')
print('-'*66)
print(f'{"RC Cosine Gap (mean)":<30} {gaps_baseline.mean():>12.6f} {gaps_rccr.mean():>12.6f} {(gaps_rccr.mean()-gaps_baseline.mean())/gaps_baseline.mean()*100:>+11.1f}%')
print(f'{"RC Cosine Gap (median)":<30} {np.median(gaps_baseline):>12.6f} {np.median(gaps_rccr):>12.6f} {(np.median(gaps_rccr)-np.median(gaps_baseline))/np.median(gaps_baseline)*100:>+11.1f}%')
print(f'{"Procrustes Disparity":<30} {proc_baseline:>12.6f} {proc_rccr:>12.6f} {(proc_rccr-proc_baseline)/proc_baseline*100:>+11.1f}%')

In [ ]:
# Visualization

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(18, 12))
gs = gridspec.GridSpec(2, 3, hspace=0.35, wspace=0.3)

# --- Panel A: RCCR Training Loss ---
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(range(1, len(rccr_loss_history)+1), rccr_loss_history, 'o-', color='#e74c3c', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('RCCR Loss (MSE)')
ax1.set_title('(A) RCCR Training Loss', fontweight='bold')
ax1.grid(True, alpha=0.3)

# --- Panel B: RC Cosine Gap Distribution ---
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(gaps_baseline, bins=40, alpha=0.6, color='#3498db', label='Baseline', density=True)
ax2.hist(gaps_rccr, bins=40, alpha=0.6, color='#e74c3c', label='RCCR', density=True)
ax2.axvline(gaps_baseline.mean(), color='#3498db', linestyle='--', linewidth=2)
ax2.axvline(gaps_rccr.mean(), color='#e74c3c', linestyle='--', linewidth=2)
ax2.set_xlabel('Cosine Distance (fwd vs RC)')
ax2.set_ylabel('Density')
ax2.set_title('(B) RC Cosine Gap Distribution', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# --- Panel C: Procrustes Comparison ---
ax3 = fig.add_subplot(gs[0, 2])
bars = ax3.bar(['Baseline', 'RCCR'], [proc_baseline, proc_rccr],
               color=['#3498db', '#e74c3c'], width=0.5)
ax3.set_ylabel('Procrustes Disparity')
ax3.set_title('(C) Procrustes: Fwd vs RC', fontweight='bold')
for bar, val in zip(bars, [proc_baseline, proc_rccr]):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f'{val:.4f}', ha='center', fontweight='bold')
ax3.grid(True, alpha=0.3, axis='y')

# --- Panel D: Composite Stability by Perturbation ---
ax4 = fig.add_subplot(gs[1, 0:2])
perts = df[df['model']=='DNABERT2_baseline']['perturbation'].tolist()
comp_base = df[df['model']=='DNABERT2_baseline']['composite'].tolist()
comp_rccr = df[df['model']=='DNABERT2_RCCR']['composite'].tolist()

x_pos = np.arange(len(perts))
w = 0.35
ax4.bar(x_pos - w/2, comp_base, w, label='Baseline', color='#3498db', alpha=0.8)
ax4.bar(x_pos + w/2, comp_rccr, w, label='RCCR', color='#e74c3c', alpha=0.8)
ax4.set_xticks(x_pos)
ax4.set_xticklabels(perts, rotation=20, ha='right')
ax4.set_ylabel('Composite Stability')
ax4.set_title('(D) Shesha Composite Stability: Baseline vs RCCR', fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3, axis='y')

# --- Panel E: Delta (RCCR - Baseline) ---
ax5 = fig.add_subplot(gs[1, 2])
deltas = [r - b for b, r in zip(comp_base, comp_rccr)]
colors = ['#27ae60' if d > 0 else '#c0392b' for d in deltas]
ax5.barh(perts, deltas, color=colors)
ax5.axvline(0, color='k', linewidth=0.8)
ax5.set_xlabel('Delta Composite (RCCR - Baseline)')
ax5.set_title('(E) RCCR Effect on Stability', fontweight='bold')
ax5.grid(True, alpha=0.3, axis='x')

plt.suptitle('DNABERT-2: Effect of RCCR on Geometric Tax', fontsize=14, fontweight='bold', y=1.01)
plt.savefig(f'{RESULTS_DIR}/dnabert2_rccr_{PHASE}.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved to {RESULTS_DIR}/dnabert2_rccr_{PHASE}.png')